# Basic usage of LaTeX parser-interpreter 

Anton Antonov   
March 2026

----

## Setup

In [20]:
use experimental :rakuast;

use LaTeX::Grammar;
use CortexJS;

If the CortexJS commands below do not work evaluate the following magic cell (uncomment first):

In [21]:
# #%bash
# npm install @cortex-js/compute-engine

()

----

## Translation examples

Translate LaTeX to [AsciiMath](https://asciimath.org):

In [22]:
latex-interpret('\e^{i * \pi} = 0', actions => 'AsciiMath')

e^(i*pi)=0

Translate LaTeX to [MathJSON](https://mathlive.io/math-json/):

In [23]:
latex-interpret('\e^{i * \pi} = 0', actions => 'MathJSON');

[Equal [Power e [Multiply i pi]] 0]

Translate LaTeX to [MathML](https://en.wikipedia.org/wiki/MathML) and render as HTML:

In [24]:
#%html
latex-interpret('\e^{i * \pi} = 0', actions => 'MathML');

e i × pi = 0

Translate LaTeX to [Raku(AST)](https://docs.raku.org/type/RakuAST):

In [25]:
latex-interpret('\sum_{n=1}^{10} n^2', actions => 'RakuAST').DEPARSE

[+] (1 .. 10).map(-> $n! { ($n ** 2) })

Translate LaTeX to [Wolfram Language (WL)](https://en.wikipedia.org/wiki/Wolfram_Language):

In [26]:
latex-interpret('\e^{i * \pi} = 0', actions => 'WL');

Equal[Power[e,Times[i,pi]],0]

Tabulate different to-formats:

In [27]:
#%html
my @formulas = (
    '\\int _{0} ^{1} x^{2} d x',
    '\\sum_{n=1}^{10} n^2',
    '\\lim_{x\to0} \\frac{\\sin(x)}{x}',
);
my @targets = <AsciiMath MathJSON MathML WL>;

my @res = do for @formulas -> $fm {
   [Formula => $fm, |@targets.map({ $_ => latex-interpret($fm, actions => $_) })].Hash
}

#@res ==> to-html-table(field-names => ['Formula', |@targets])
@res.map(*<MathML>) ==> to-html-table()

∫01x2dx,∑n=110n2,limx→0sin(x)x


In [28]:
select-columns(@res, <Formula MathJSON WL>)
==> to-pretty-table(field-names => <Formula MathJSON WL>, align => 'l')

+--------------------------------+---------------------------------------------------+-------------------------------------------+
| Formula                        | MathJSON                                          | WL                                        |
+--------------------------------+---------------------------------------------------+-------------------------------------------+
| \int _{0} ^{1} x^{2} d x       | Integrate Function Block Power x 2 x Limits x 0 1 | Integrate[Power[x,2],{x,0,1}]             |
| \sum_{n=1}^{10} n^2            | Sum Power n 2 Limits n 1 10                       | Sum[Power[n,2],{n,1,10}]                  |
| \lim_{x\to0} \frac{\sin(x)}{x} | Limit Function Block Divide Sin x x x 0           | Limit[Times[ Sin[x] , Power[x, -1]],x->0] |
+--------------------------------+---------------------------------------------------+-------------------------------------------+

In [29]:
#% html
[
    '4 x^2 + 12 x + 9',
    '4 * x^2 + 12 * x + 9',
    '\sqrt{4 * x^2 + 12 * x + 9}',
]
andthen $_.map({ %(expr => "latex«$_»", Cortex-JS => parse-latex($_).raku, 'LaTeX::Grammar' => latex-interpret($_).raku) })
andthen .&to-html(field-names => <expr Cortex-JS LaTeX::Grammar>, align => 'left')
andthen .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g)

expr,Cortex-JS,LaTeX::Grammar
4×x2+12×x+9,"$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","$[""Add"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""]], 9]"
4×x2+12×x+9,"$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","$[""Add"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""]], 9]"
4×x2+12×x+9,"$[""Sqrt"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]]","$[""Root"", [""Add"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""]], 9], 2]"


In [30]:
#%html

my @formulas = (
'\frac{-1214}{117}',
'\\sqrt{4 * x^2 + 12 * x + 9}',
'\\int_{0}^{1} x^{2} d x',
'\\sum_{n=1}^{10} n^2',
'\\lim_{x\\to0} \\frac{\\sin(x)}{x}',
'\\log_{5} x',
'\log\left( \frac{x+1}{x-1} \right)'
);
my @targets = <AsciiMath WL MathJSON>;

my @res = do for @formulas -> $fm {
    [Formula => $fm, MathML => "latex«$fm»", |@targets.map({ $_ => latex-interpret($fm, actions => $_).raku })].Hash
}

@res 
==> to-html(field-names => ['Formula', 'MathML', |@targets], align => 'left')
==> { .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g) }()
==> { .subst('"', :g) }()

Formula,MathML,AsciiMath,WL,MathJSON
\frac{-1214}{117},-1214117,"""-1214/117""","""Rational[-1214,117]""","$[""Rational"", -1214, 117]"
\sqrt{4 * x^2 + 12 * x + 9},4×x2+12×x+9,"""sqrt((4*x^2+12*x)+9)""","""Sqrt[Plus[Plus[Times[4,Power[x,2]],Times[12,x]],9]]""","$[""Root"", [""Add"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""]], 9], 2]"
\int_{0}^{1} x^{2} d x,∫01x2dx,"""int x^2""","""Integrate[Power[x,2],\{x,0,1}]""","$[""Integrate"", [""Function"", [""Block"", [""Power"", ""x"", 2]], ""x""], [""Limits"", ""x"", 0, 1]]"
\sum_{n=1}^{10} n^2,∑n=110n2,"""sum n^2""","""Sum[Power[n,2],\{n,1,10}]""","$[""Sum"", [""Power"", ""n"", 2], [""Limits"", ""n"", 1, 10]]"
\lim_{x\to0} \frac{\sin(x)}{x},limx→0sin(x)x,"""lim_(x->0) sin(x)/x""","""Limit[Times[ Sin[x] , Power[x, -1]],x->0]""","$[""Limit"", [""Function"", [""Block"", [""Divide"", [""Sin"", ""x""], ""x""]], ""x""], 0]"
\log_{5} x,log5×x,"""log_5*x""","""Times[Subscript[log,5],x]""","$[""Multiply"", [""Subscript"", ""log"", 5], ""x""]"
\log\left( \frac{x+1}{x-1} \right),log(left×x+1x-1×right),"""log(left*(x+1)/(x-1)*right)""","""Log[Times[left,Times[Rational[Plus[x,1],Plus[x,Times[-1,1]]],right]]]""","$[""Log"", [""Multiply"", ""left"", [""Multiply"", [""Divide"", [""Add"", ""x"", 1], [""Subtract"", ""x"", 1]], ""right""]]]"


---

## Comparison with CortexJS LaTeX conversion to MathJSON

The LaTeX expressions can be converted to MathJSON using the Cortex-JS functions or by the Raku package ["LaTeX::Grammar"](https://raku.land/zef:antononcube/LaTeX::Grammar). Here is an example with the latter:

In [31]:
#% html
my @res = do for @formulas -> $fm {
    [Formula => $fm, MathML => "latex«$fm»", 'LaTeX::Grammar' => latex-interpret($fm).raku, CortexJS => parse-latex($fm).raku].Hash
}

@res 
==> to-html(field-names => ['Formula', 'MathML', 'LaTeX::Grammar', 'CortexJS'], align => 'left')
==> { .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g) }()
==> { .subst('"', :g) }()

Formula,MathML,LaTeX::Grammar,CortexJS
\frac{-1214}{117},-1214117,"$[""Rational"", -1214, 117]","$[""Rational"", -1214, 117]"
\sqrt{4 * x^2 + 12 * x + 9},4×x2+12×x+9,"$[""Root"", [""Add"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""]], 9], 2]","$[""Sqrt"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]]"
\int_{0}^{1} x^{2} d x,∫01x2dx,"$[""Integrate"", [""Function"", [""Block"", [""Power"", ""x"", 2]], ""x""], [""Limits"", ""x"", 0, 1]]","$[""Integrate"", [""Function"", [""Block"", [""Power"", ""x"", 2]], ""x""], [""Limits"", ""x"", 0, 1]]"
\sum_{n=1}^{10} n^2,∑n=110n2,"$[""Sum"", [""Power"", ""n"", 2], [""Limits"", ""n"", 1, 10]]","$[""Sum"", [""Power"", ""n"", 2], [""Limits"", ""n"", 1, 10]]"
\lim_{x\to0} \frac{\sin(x)}{x},limx→0sin(x)x,"$[""Limit"", [""Function"", [""Block"", [""Divide"", [""Sin"", ""x""], ""x""]], ""x""], 0]","$[""Limit"", [""Function"", [""Block"", [""Divide"", [""Sin"", ""x""], ""x""]], ""x""], 0]"
\log_{5} x,log5×x,"$[""Multiply"", [""Subscript"", ""log"", 5], ""x""]","$[""Log"", ""x"", 5]"
\log\left( \frac{x+1}{x-1} \right),log(left×x+1x-1×right),"$[""Log"", [""Multiply"", ""left"", [""Multiply"", [""Divide"", [""Add"", ""x"", 1], [""Subtract"", ""x"", 1]], ""right""]]]","$[""Log"", [""Divide"", [""Add"", ""x"", 1], [""Add"", ""x"", -1]]]"
